# APs plots in the time domain

In [3]:
CORRELATION_ANALYSIS = False
MAX_HOUR_IN_SECONDS = 60*60*4

Required files:
- TTL times
- TTL labels
- APs data (can be read using TTL times)
    - Spike times
    - Spike Clusters
    - Cluster info
    - Templates   
- Sleep/Wake labels

In [35]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd

import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

main_folder = r'\\132.66.34.210\Pixel1\1_auditory_neuropixels_BarakH\20240912_C11_T1_NP2_-10dB_g0'
SleepWakeLabels_fname = 'labels_All.mat'
TTL_times_fname = 'TTL_times_adj.txt'
TTL_labels_fname = 'TTL_labels.txt'
AP_spike_times_fname = r'spike_times_sec.npy'
AP_spike_clusters_fname = r'spike_clusters.npy'
AP_cluster_info_fname = r'cluster_info'
AP_templates_fname = r'templates.npy'

ClusterInfo_path = find_files(main_folder, AP_cluster_info_fname)[0]
Templates_path = find_files(main_folder, AP_templates_fname)[0]
SpikeTimes_path = find_files(main_folder, AP_spike_times_fname)[0]
SpikeClusters_path = find_files(main_folder, AP_spike_clusters_fname)[0]
TTL_labels_path = find_files(main_folder, TTL_labels_fname)[0]
TTL_times_path = find_files(main_folder, TTL_times_fname)[0]
SleepWakeLabels_path = find_files(main_folder, SleepWakeLabels_fname)[0]

In [36]:
Templates = np.load(Templates_path)
SpikeTimes = np.squeeze(np.load(SpikeTimes_path))
SpikeClusters = np.load(SpikeClusters_path)
ClusterInfo = pd.read_csv(ClusterInfo_path, sep='\t')

TTL_times = np.loadtxt(TTL_times_path)
TTL_labels = np.loadtxt(TTL_labels_path)

SleepWakeLabels =  scipy.io.loadmat(SleepWakeLabels_path)
SleepWakeLabels = np.squeeze(SleepWakeLabels['labels'])
SleepWakeTimes = np.linspace(0, MAX_HOUR_IN_SECONDS, len(SleepWakeLabels))  # the stop value is set by the time I used for manual scoring

## Plot Single Unit

In [49]:
SoundType = 101
cluster_pick = 234
ClusterInfo_clusterpick = ClusterInfo[ClusterInfo['cluster_id'] == cluster_pick]

if SoundType == 100:
    TTL_picked = (TTL_labels > 100) & (TTL_labels < 200)
elif SoundType == 400:
    TTL_picked = (TTL_labels > 400) & (TTL_labels < 600)
elif SoundType == 1300:
    TTL_picked = (TTL_labels > 1300) & (TTL_labels < 1400)
elif SoundType == 1400:
    TTL_picked = (TTL_labels > 1400) & (TTL_labels < 1500)
else:
    TTL_picked = (TTL_labels == SoundType)
print(f'SoundType {SoundType} - Number of trials: {np.sum(TTL_picked)}')

In [50]:
cur_SoundType_trialsTimes = TTL_times[TTL_picked]
cur_SoundType_trialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes > 1]

closest_SleepWakeTimes = []
for curTTLtime in cur_SoundType_trialsTimes:
    closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))

cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]

All_TrialsTimes = cur_SoundType_trialsTimes
First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes < 30*60]
print(f'First30min - Number of trials: {len(First30min_TrialsTimes)}')
Last30min_TrialsTimes = cur_SoundType_trialsTimes[(MAX_HOUR_IN_SECONDS - 30*60) < cur_SoundType_trialsTimes]
print(f'Last30min - Number of trials: {len(Last30min_TrialsTimes)}')
Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
print(f'Sleep - Number of trials: {len(Sleep_TrialsTimes)}')
Wake_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 2]
print(f'Wake - Number of trials: {len(Wake_TrialsTimes)}')

In [51]:
Picked_TrialsTimes = All_TrialsTimes
tstep = 4.5 # Take 4.5 seconds after -1 from onset (3.5 seconds after stimuli)
Trials_read_times_start = Picked_TrialsTimes - 1 # Take one second before Stimuli Onset
Trials_read_times_end = Trials_read_times_start + tstep # End of trial

Trials_label = list(range(0,len(Picked_TrialsTimes)))

Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
gaps_ind = np.argwhere((Trials_sleep[1:] - Trials_sleep[:-1])>1)[:,0]
Sleep_end = np.append(Trials_sleep[gaps_ind], Trials_sleep[-1])
Sleep_start = np.append(Trials_sleep[0], Trials_sleep[gaps_ind+1])

Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
gaps_ind = np.argwhere((Trials_Wake[1:] - Trials_Wake[:-1])>1)[:,0]
Wake_end = np.append(Trials_Wake[gaps_ind], Trials_Wake[-1])
Wake_start = np.append(Trials_Wake[0], Trials_Wake[gaps_ind+1])

Filter Spikes within trials

In [52]:
mask = np.zeros_like(SpikeTimes, dtype=bool)
SpikeTrials = np.ones_like(SpikeTimes) * np.nan
SpikeTimes_inTrials = np.ones_like(SpikeTimes) * np.nan
for ind, (start, end) in enumerate(zip(Trials_read_times_start, Trials_read_times_end)):
    cur_mask = (SpikeTimes >= start) & (SpikeTimes <= end)
    SpikeTimes_inTrials[cur_mask] = SpikeTimes[cur_mask] - start - 1
    SpikeTrials[cur_mask] = ind
    mask |= cur_mask

mask = np.squeeze(mask)
SpikeTimes_inTrials = SpikeTimes_inTrials[mask]
SpikeClusters_inTrials = SpikeClusters[mask]
SpikeTrials_inTrials = SpikeTrials[mask]

Filter specific Cluster

In [53]:
SpikeTimes_inTrials_plot = SpikeTimes_inTrials[SpikeClusters_inTrials == cluster_pick]
SpikeTrials_inTrials_plot = SpikeTrials_inTrials[SpikeClusters_inTrials == cluster_pick]

t_vec = np.arange(np.min(SpikeTimes_inTrials_plot), np.max(SpikeTimes_inTrials_plot), 0.05)
hist, bin_edges = np.histogram(SpikeTimes_inTrials_plot, bins=t_vec, density=False)
hist_FR = hist / (t_vec[1] - t_vec[0]) / len(Picked_TrialsTimes)

ind_sleep = np.isin(SpikeTrials_inTrials_plot, Trials_sleep)
SpikeTimes_inTrials_plot_sleep = SpikeTimes_inTrials_plot[ind_sleep]
SpikeTrials_inTrials_plot_sleep = SpikeTrials_inTrials_plot[ind_sleep]
hist_sleep, _ = np.histogram(SpikeTimes_inTrials_plot_sleep, bins=t_vec, density=False)
hist_FR_sleep = hist_sleep / (t_vec[1] - t_vec[0]) / len(Trials_sleep)

ind_wake = np.isin(SpikeTrials_inTrials_plot, Trials_Wake)
SpikeTimes_inTrials_plot_wake = SpikeTimes_inTrials_plot[ind_wake]
SpikeTrials_inTrials_plot_wake = SpikeTrials_inTrials_plot[ind_wake]
hist_wake, _ = np.histogram(SpikeTimes_inTrials_plot_wake, bins=t_vec, density=False)
hist_FR_wake = hist_wake / (t_vec[1] - t_vec[0]) / len(Trials_Wake)

Plot Raster plot

In [54]:
%matplotlib inline
# %matplotlib qt
display(ClusterInfo_clusterpick)

def draw_recs_on_plot(ax, rec_start_inds, rec_end_inds, color_rec):
    for start_ind, end_ind in zip(rec_start_inds, rec_end_inds):
        start_trial = start_ind
        n_trials = end_ind - start_ind
        y_vec = np.linspace(start_trial, end_ind, n_trials)
        x_vec = np.ones_like(y_vec) * -0.99
        # Add a shaded rectangle
        ax.plot(x_vec, y_vec, color=color_rec, linewidth=10)
        # rect = Rectangle((-1, start_trial), 4.5, n_trials, alpha=0.3, color=color_rec)
        # ax.add_patch(rect)
        
plt.figure(figsize=(18, 9), tight_layout=True)
ax = plt.subplot2grid((5, 1), (1, 0), rowspan=4, colspan=1)
# ax.scatter(SpikeTimes_inTrials_plot, SpikeTrials_inTrials_plot, marker='|', color='r', s=10, alpha=0.5)
ax.scatter(SpikeTimes_inTrials_plot_sleep, SpikeTrials_inTrials_plot_sleep, marker='|', color='r', s=10, alpha=0.5)
ax.scatter(SpikeTimes_inTrials_plot_wake, SpikeTrials_inTrials_plot_wake, marker='|', color='g', s=10, alpha=0.5)
ax.axvline(x=0, color='k', linestyle='--', linewidth=1)
ax.set_xlabel('Time [Sec]')
ax.set_ylabel('Trials')
draw_recs_on_plot(ax, Sleep_start, Sleep_end, 'red')
draw_recs_on_plot(ax, Wake_start, Wake_end, 'green')
ax.set_xlim([min(t_vec), max(t_vec)])
ax.set_ylim([min(Trials_label), max(Trials_label)])


ax2 = plt.subplot2grid((5, 1), (0, 0), rowspan=1, colspan=1)
ax2.bar(t_vec[:-1], hist_FR, width=t_vec[1] - t_vec[0], alpha=0.2, color='k',edgecolor='k', align='edge')
ax2.plot(t_vec[1:], hist_FR_sleep, color='r')
ax2.plot(t_vec[1:], hist_FR_wake, color='g')
# ax2.bar(t_vec[:-1], hist_FR_sleep, width=t_vec[1] - t_vec[0], alpha=0.8, color='r')
# ax2.bar(t_vec[:-1], hist_FR_wake, width=t_vec[1] - t_vec[0], alpha=0.5, color='g')
ax2.set_xticks([])
ax2.set_ylabel('FR [Spikes/Sec]')
ax2.set_xlim([min(t_vec), max(t_vec)])

axins = ax2.inset_axes([0.9, 0.5, 0.1, 0.5])
axins.plot(np.squeeze(Templates[cluster_pick, :, ClusterInfo_clusterpick['ch']]))
axins.set_xticks([])
axins.set_yticks([])
axins.axis('off')

## Correlation Analysis

In [ ]:
def filter_by_soundType_single(SoundType, TTL_times, SleepWakeTimes, SleepWakeLabels, SpikeTimes, SpikeClusters):
    if SoundType == 100:
        TTL_picked = (TTL_labels > 100) & (TTL_labels < 200)
    elif SoundType == 400:
        TTL_picked = (TTL_labels > 400) & (TTL_labels < 600)
    elif SoundType == 1300:
        TTL_picked = (TTL_labels > 1300) & (TTL_labels < 1400)
    elif SoundType == 1400:
        TTL_picked = (TTL_labels > 1400) & (TTL_labels < 1500)
    else:
        TTL_picked = (TTL_labels == SoundType)
        
    print(f'SoundType {SoundType} - Number of trials: {np.sum(TTL_picked)}')
    cur_SoundType_trialsTimes = TTL_times[TTL_picked]
    
    cur_SoundType_trialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes > 1]
    
    closest_SleepWakeTimes = []
    for curTTLtime in cur_SoundType_trialsTimes:
        closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))
    
    cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]
    
    All_TrialsTimes = cur_SoundType_trialsTimes
    First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes < 30*60]
    Last30min_TrialsTimes = cur_SoundType_trialsTimes[ (MAX_HOUR_IN_SECONDS - 30*60) < cur_SoundType_trialsTimes]
    Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
    Wake_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 2]
    
    Picked_TrialsTimes = All_TrialsTimes
    
    tstep = 4.5 # Take 4.5 seconds after -1 from onset (3.5 seconds after stimuli)
    Trials_read_times_start = Picked_TrialsTimes - 1 # Take one second before Stimuli Onset
    Trials_read_times_end = Trials_read_times_start + tstep # End of trial
    
    Trials_label = list(range(0,len(Picked_TrialsTimes)))
    Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
    Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
    
    mask = np.zeros_like(SpikeTimes, dtype=bool)
    SpikeTrials = np.ones_like(SpikeTimes) * np.nan
    SpikeTimes_inTrials = np.ones_like(SpikeTimes) * np.nan
    for ind, (start, end) in enumerate(zip(Trials_read_times_start, Trials_read_times_end)):
        cur_mask = (SpikeTimes >= start) & (SpikeTimes <= end)
        SpikeTimes_inTrials[cur_mask] = SpikeTimes[cur_mask] - start - 1
        SpikeTrials[cur_mask] = ind
        mask |= cur_mask
    
    mask = np.squeeze(mask)
    SpikeTimes_inTrials = SpikeTimes_inTrials[mask]
    SpikeClusters_inTrials = SpikeClusters[mask]
    SpikeTrials_inTrials = SpikeTrials[mask]
    
    return SpikeTimes_inTrials, SpikeClusters_inTrials, SpikeTrials_inTrials, Trials_sleep, Trials_Wake, Trials_label

def filter_by_cluster_single(cluster_pick, ClusterInfo, SpikeTimes_inTrials, SpikeTrials_inTrials, SpikeClusters_inTrials, Trials_sleep, Trials_Wake, Trials_label):
    t_vec = np.arange(-1, 3.5, 0.1)
    
    Spikes_array = np.zeros((len(Trials_label), len(t_vec)))

    SpikeTimes_inTrials_plot = SpikeTimes_inTrials[SpikeClusters_inTrials == cluster_pick]
    SpikeTrials_inTrials_plot = SpikeTrials_inTrials[SpikeClusters_inTrials == cluster_pick].astype(int)
    
    time_bin_indices = np.searchsorted(t_vec, SpikeTimes_inTrials_plot) - 1
    time_bin_indices = np.clip(time_bin_indices, 0, len(t_vec) - 1)
    Spikes_array[SpikeTrials_inTrials_plot, time_bin_indices] = 1
    return Spikes_array

if CORRELATION_ANALYSIS:
    df = pd.DataFrame({"SoundType":[], "Channel": [], "Cluster": [], "Spikes_All": [], "Trials_sleep": [], "Trials_wake": []})
    
    SoundTypes = [101, 102, 400, 1300, 1401, 1402, 1403]
    
    ClusterInfo.sort_values('fr', ascending=False, inplace=True)
    max_fr = np.max(ClusterInfo['fr'])
    ClusterInfo_top = ClusterInfo.copy()
    # ClusterInfo_top = ClusterInfo_top[(ClusterInfo_top['fr'] >= max_fr/10) & (ClusterInfo_top['ch'] < 300)]
    # ClusterInfo_top = ClusterInfo_top[(ClusterInfo_top['KSLabel'] == 'good') & (ClusterInfo_top['ch'] < 300)]
    ClusterInfo_top = ClusterInfo_top[(ClusterInfo_top['fr'] >= max_fr/10) & (ClusterInfo_top['KSLabel'] == 'good') & (ClusterInfo_top['ch'] < 300)]
    
    cluster_picks = ClusterInfo_top['cluster_id'].unique()
    
    for SoundType in SoundTypes:
        SpikeTimes_inTrials, SpikeClusters_inTrials, SpikeTrials_inTrials, Trials_sleep, Trials_Wake, Trials_label = filter_by_soundType_single(SoundType, TTL_times, SleepWakeTimes, SleepWakeLabels, SpikeTimes, SpikeClusters)
        for cluster_pick in cluster_picks:
            ClusterInfo_clusterpick = ClusterInfo[ClusterInfo['cluster_id'] == cluster_pick]
            Spikes_array = filter_by_cluster_single(cluster_pick, ClusterInfo, SpikeTimes_inTrials, SpikeTrials_inTrials, SpikeClusters_inTrials, Trials_sleep, Trials_Wake, Trials_label)
            new_row_df = pd.DataFrame([{"SoundType":int(SoundType), "Channel": int(ClusterInfo_clusterpick['ch'].iloc[0]), "Cluster": cluster_pick, "Spikes_All": Spikes_array, "Trials_sleep": Trials_sleep, "Trials_wake": Trials_Wake}])
            df = pd.concat([df, new_row_df], ignore_index=True)

In [ ]:
if CORRELATION_ANALYSIS:
    
    def calculate_corr_mat(df_usv, picked_trials, t_vec, analysis_type):
        # Initialize a list to store the correlation vectors
        correlation_mat = np.zeros((len(df_usv), len(df_usv)))
    
        for i in range(len(df_usv)):
            mat_1 = np.squeeze(df_usv.loc[:,'Spikes_All'].iloc[i])
            mat_1 = mat_1[picked_trials, :]
            # mat_1 = mat_1[:, (t_vec>0.1) & (t_vec<2.5)]
            
            for j in range(i, len(df_usv)):
                # if i == j:
                #     continue
                mat_2 = np.squeeze(df_usv.loc[:,'Spikes_All'].iloc[j])
                mat_2 = mat_2[picked_trials, :]
                # mat_2 = mat_2[:, (t_vec>0.1) & (t_vec<2.5)]
              
                if analysis_type == 'noise':
                    '''Compute the correlation coefficient for each trial'''
                    trial_correlations = []
                    for trial in range(mat_1.shape[0]):
                        x = mat_1[trial]
                        y = mat_2[trial]
                        if (np.std(x) == 0) or (np.std(y) == 0):
                            trial_corr = np.nan
                        else:
                            trial_corr = np.corrcoef(x, y)[0, 1]
                        trial_correlations.append(trial_corr)
                    '''Append the list of trial correlation coefficients to the main list'''
                    correlation_mat[i, j] = np.nanmean(trial_correlations)
                else:
                    '''Compute the correlation for mean of all trials'''
                    correlation_mat[i, j] = np.corrcoef(np.mean(mat_1, axis=0), np.mean(mat_2, axis=0))[0, 1]
                    
            
        correlation_mat = correlation_mat + correlation_mat.T - np.diag(np.diag(correlation_mat))
        return correlation_mat
    
    df_usv = df.copy()
    metric_axis = 'Channel'
    check_sound = 1403
    t_vec = np.arange(-1, 3.5, 0.1)
    analysis_type = 'signal'  # 'signal', 'noise'
    
    df_usv.sort_values(f'{metric_axis}', ascending=False, inplace=True)
    df_usv = df_usv[df_usv['SoundType'] == check_sound].reset_index(drop=True)
    sleep_trials = np.squeeze(df_usv.loc[:, 'Trials_sleep'].iloc[0])
    wake_trials = np.squeeze(df_usv.loc[:, 'Trials_wake'].iloc[0])
    all_trials = list(range(0,df_usv.loc[:,'Spikes_All'].iloc[0].shape[0]))
    
    picked_trials = all_trials
    # correlation_mat = calculate_corr_mat(df_usv, sleep_trials, t_vec, analysis_type) - calculate_corr_mat(df_usv, wake_trials, t_vec, analysis_type)
    correlation_mat = calculate_corr_mat(df_usv, picked_trials, t_vec, analysis_type)
   

In [ ]:
if CORRELATION_ANALYSIS:
    from scipy.ndimage import gaussian_filter

    # %matplotlib inline
    %matplotlib qt
    labels = np.array(df_usv[f'{metric_axis}'].to_list())
    ticks_locs = np.arange(0, correlation_mat.shape[0], 1)
    
    plt.figure(figsize=(10, 10), layout='constrained')
    ind_1 = 6
    ind_2 = 7
    ch_1 = df_usv.loc[:,'Channel'].iloc[ind_1]
    ch_2 = df_usv.loc[:,'Channel'].iloc[ind_2]
    clus_1 = df_usv.loc[:,'Cluster'].iloc[ind_1]
    clus_2 = df_usv.loc[:,'Cluster'].iloc[ind_2]
    mat_1 = df_usv.loc[:,'Spikes_All'].iloc[ind_1]
    mat_2 = df_usv.loc[:,'Spikes_All'].iloc[ind_2]
    mat_1 = mat_1[picked_trials]
    mat_2 = mat_2[picked_trials]
    
    ax1 = plt.subplot2grid((7, 4), (0, 0), rowspan=1, colspan=2)
    ax1.plot(t_vec, np.mean(mat_1, axis=0)/(t_vec[1] - t_vec[0]))
    ax1.set_title(f'ch - {ch_1} clus - {clus_1}')
    ax1.set_xlim([t_vec[0], t_vec[-1]])
    ax1 = plt.subplot2grid((7, 4), (1, 0), rowspan=2, colspan=2)
    ax1.imshow(mat_1, cmap=plt.get_cmap('RdBu'), vmin=0, vmax=1, aspect='auto')
    ax1.axis('off')
    ax2 = plt.subplot2grid((7, 4), (0, 2), rowspan=1, colspan=2)
    ax2.plot(t_vec, np.mean(mat_2, axis=0)/(t_vec[1] - t_vec[0]))
    ax2.set_title(f'ch - {ch_2} clus - {clus_2}')
    ax2.set_xlim([t_vec[0], t_vec[-1]])
    ax2 = plt.subplot2grid((7, 4), (1, 2), rowspan=2, colspan=2)
    ax2.imshow(mat_2, cmap=plt.get_cmap('RdBu'), vmin=0, vmax=1, aspect='auto')
    ax2.axis('off')

    ax1 = plt.subplot2grid((7, 4), (3, 0), rowspan=4, colspan=4)
    if analysis_type == 'noise':
        im = ax1.imshow(correlation_mat, aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'), vmin=-0.1, vmax=0.1)
    else:
        im = ax1.imshow(gaussian_filter(correlation_mat,1), aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'))

    cbar = plt.colorbar(im, ax=ax1, shrink=0.6 )
    ax1.set_xticks(ticks_locs[::5], labels[::5])
    ax1.set_yticks(ticks_locs[::5], labels[::5])
    ax1.set_xlabel(f'{metric_axis}')
    ax1.set_ylabel(f'{metric_axis}')
    ax = plt.gca()

    # Minor ticks
    ax.set_xticks(np.array([ind_1,ind_1 + 1]) - 0.5, minor=True)
    ax.set_yticks(np.array([ind_2,ind_2 + 1]) - 0.5, minor=True)
    ax.grid(which='minor', color='g', linestyle='-', linewidth=1)
    
    plt.suptitle(f'{analysis_type} - Sound {check_sound} - [ind_1, ind_2]: [{ind_1},{ind_2}] Corr coeff: {round(correlation_mat[ind_1, ind_2], 3)}')
    

In [ ]:
if CORRELATION_ANALYSIS:
    # %matplotlib inline
    %matplotlib qt
    noise_correlation = calculate_corr_mat(df_usv, picked_trials, t_vec, 'noise')
    signal_correlation = calculate_corr_mat(df_usv, picked_trials, t_vec, 'signal')
    noise_correlation_phase = np.arccos(noise_correlation)
    signal_correlation_phase = np.arccos(signal_correlation)
    abs_phase_diff = np.abs(signal_correlation_phase - noise_correlation_phase)

In [ ]:
    plt.figure(figsize=(18, 6))
    plt.subplot(1, 5, 1)
    plt.imshow(noise_correlation, aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'))
    plt.subplot(1, 5, 2)
    plt.imshow(signal_correlation, aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'))
    plt.subplot(1, 5, 3)
    plt.imshow(noise_correlation_phase, aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'))
    plt.subplot(1, 5, 4)
    plt.imshow(signal_correlation_phase, aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'))
    plt.subplot(1, 5, 5)
    plt.imshow(np.rad2deg(abs_phase_diff), aspect='auto', origin='upper', cmap=plt.get_cmap('RdBu'))
    plt.colorbar()